# Cascaded-SolarNet: test + train on Colab

Clones the repo, runs the test suite, downloads InfraredSolarModules, trains the cascade and baselines, and evaluates end to end.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU**. You also need a Kaggle API token (kaggle.com -> Settings -> API -> Create New Token), which gives you a `kaggle.json`.

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())

## 1. Clone and install

In [ ]:
%cd /content
!rm -rf cascaded-cnn
!git clone https://github.com/techwack/cascaded-cnn
%cd /content/cascaded-cnn
# torch is already installed on Colab; install the rest
!pip install -q numpy scikit-learn pillow pyyaml pytest kaggle
!pip install -q -e . --no-deps

## 2. Run the tests (synthetic data, no download needed)

In [ ]:
!python -m pytest -q

## 3. Download the dataset
Paste your Kaggle API token (the `KGAT_...` string from kaggle.com -> Settings -> API -> Create New Token) into the hidden prompt. It is only kept in this Colab VM; never commit it.

In [ ]:
import os, getpass
!pip install -q -U kaggle
token = getpass.getpass("Kaggle API token: ").strip()
os.environ["KAGGLE_API_TOKEN"] = token
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
path = os.path.expanduser("~/.kaggle/access_token")
open(path, "w").write(token)
os.chmod(path, 0o600)
del token

In [ ]:
!bash scripts/download_data.sh
!python scripts/make_splits.py

## 4. Train
Each node trains on ground-truth routing. Early stopping is on val macro-F1, so this stops well before 60 epochs.

In [ ]:
!python -m solarnet.train --node all

In [ ]:
!python -m solarnet.train --node flat
!python baselines/random_forest.py

## 5. Evaluate the whole cascade on the test split

In [ ]:
!python -m solarnet.evaluate --flat

In [ ]:
import json
r = json.load(open("runs/results.json"))
print(json.dumps(r, indent=2)[:6000])

## 6. Save results
Downloads the logs and metrics (not the large checkpoints) so you can commit them or paste numbers into the report.

In [ ]:
!zip -qr results.zip runs -x "*.pt"
from google.colab import files
files.download("results.zip")